# Präambel

In [1]:
from sympy import *
from IPython.display import display, Math, Latex, Markdown
#init_session()
init_printing(use_unicode=True)

In [2]:
class dotdict(dict):
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

def md(s):
    display(Markdown(s))
    
def ausnutzung(p, stab=None):
    color = '#000'
    emoji = ''
    if p > 1:
        color = '#f00'
        emoji = '🔴'
    elif p > 0.7:
        color = '#0f0'
        emoji = '🟢'
    elif p < 0.7:
        color = '#00f'
        emoji = '🟡'
    #md(f'#### <span style="color:{color};">{int(p*100)}%</span>')
    qs = stab.querschnitt if stab else ''
    md(f'#### {emoji} {int(N(p, 2)*100)}% ({qs})')
    return p


def note(s):
    md(f'<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">{s}</span></div>')
ausnutzung(13.37)
ausnutzung(0.238)
N(0.238, 2) * 100

#### 🔴 1338% ()

#### 🟡 23% ()

In [104]:
%run querschnitte.ipynb

In [4]:
from dataclasses import dataclass
from decimal import Decimal
from functools import partial

@dataclass
class Schnittgroeszen:
    N: float = None
    V: float = None
    My: float = None
    Mz: float = None
    ψ: float = None
    
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    M = lambda self, achse: self.um_achse('M', achse)


    
@dataclass
class Stahlguete:
    guete: str
    E: int
    fy: float
    fu: float
    
@dataclass
class Stab(Querschnitt, Stahlguete):
    l: float
    sg: dict
    Lcry: float = None
    Lcrz: float = None
    

    def __init__(self, guete, QS, **kwargs):
        sg = kwargs.pop('sg', None)
        if isinstance(sg, dict):
            self.sg = sg
        else:
            self.sg = {None: sg}
            
        for k,v in kwargs.items():
            self.__setattr__(k, v)
            
        Querschnitt.__init__(self, **QS.__dict__)
        Stahlguete.__init__(self, **guete.__dict__)
        
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    Lcr = lambda self, achse: self.um_achse('Lcr', achse)

S235 = Stahlguete(guete='S235', E=21000, fy=23.5, fu=36)


# Nachweis-Funktionen

## Querschnitte

### Zug

In [5]:
def zug(stab, sg=None, A_net=None):
    γ_M0 = 1
    γ_M2 = 1.25
    N_plRd = (stab.A * stab.fy)/γ_M0
    N_uRd = (0.9 * A_net * stab.fu)/γ_M2 if A_net else oo
    last = stab.sg[sg]

    return ausnutzung(N(last.N/min(N_plRd, N_uRd), 3), stab)

### Druck

In [6]:
def druck(stab, sg=None):
    γ_M0 = 1
    N_cRd = (stab.A * stab.fy)/γ_M0
    last = stab.sg[sg]
    
    return ausnutzung(N(last.N/N_cRd, 3), stab)

### Biegung

In [7]:
def biegung(um_achse, stab, sg=None):
    γ_M0 = 1
    h = stab.h if um_achse == 'y' else stab.b
    W_elmin = stab.I(um_achse)/(h/2)
    M_elRd = (W_elmin * stab.fy)/γ_M0
    
    last = stab.sg[sg]
    return ausnutzung(N(last.My/M_elRd, 3), stab)

biegung_y = partial(biegung, 'y')
biegung_z = partial(biegung, 'z')

### Querkraft

In [8]:
def querkraft(um_achse, stab, sg=None):
    γ_M0 = 1
    V_plRd = stab.Av(um_achse) * (f_y/(sqrt(3) * γ_M0))

    last = stab.sg[sg]
    return ausnutzung(N(last.V/V_plRd, 3), stab)

querkraft_y = partial(querkraft, 'y')
querkraft_z = partial(querkraft, 'z')

### Kombi: M + N

In [9]:
def kombi_M_N(um_achse, stab, sg=None):
    γ_M0 = 1
    V_plRd = stab.Av(um_achse) * (stab.fy/(sqrt(3) * γ_M0))
    last = stab.sg[sg]
    if (last.V / V_plRd) <= 0.5:
        md('Kombinierter Nachweis nicht Notwendig')
        md('Zug/Druck')
        zug(stab, sg)
        md('Biegung')
        biegung(um_achse, stab, sg)
    else:
        return ausnutzung(druck(stab, sg) + biegung(um_achse, stab, sg))

## Bauteil

In [10]:
γ_M1 = 1 #TODO tripple-check

def λ1(stab):
    return pi * sqrt(stab.E/stab.fy)

def λyz(um_achse, stab):
    return stab.Lcr(um_achse)/(stab.i(um_achse) * λ1(stab))

def KL2α(KL: str):
    return {'a0': 0.13, 'a': 0.21, 'b': 0.34, 'c': 0.49, 'd': 0.76}.get(KL)
    
    
def χyz(um_achse, stab):
    λ = λyz(um_achse, stab)
    α = KL2α(stab.KL(um_achse))
    Φ = 0.5* (1 + α * (λ-0.2) + λ**2)
    return min(1, 1/(Φ + sqrt(Φ**2 - λ**2)))

### Biegeknicken

In [41]:
def biege_knicken(um_achse, stab, sg=None):
    last = stab.sg[sg]
    assert(stab.Lcr(um_achse))
    assert(last.ψ is not None or last.My == 0)
    χ = χyz(um_achse, stab)
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) ## CHECK past?
    My_Rd =  W_y * (stab.fy / γ_M1) # W_y passt?

    n_y = last.N / (χ * N_Rd)
    k_y = min((1 + (λyz(um_achse, stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    α = 0.6 if um_achse == 'z' else 1

    C_my = max(0.6 + 0.4 * (last.ψ or 0),
               0,4)
    res = (last.N/(χ * N_Rd)) + (α * k_y * (C_my * last.My)/My_Rd)
    return ausnutzung(N(res, 5), stab)

biege_knicken_y = partial(biege_knicken, 'y')
biege_knicken_z = partial(biege_knicken, 'z')
note('Passt W_Y berechnung?')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">Passt W_Y berechnung?</span></div>

### Biegedrillknicken

In [12]:
def χLT(stab): ## Gem. EC3 (6.3.2.3)
    λz = λyz('z', stab)
    
    k_c = 1 # Auf der sicheren Seite
    k_p = 0.9 * 1 / (1 + (1/20) * (λz/(stab.h/stab.tf))**2)**0.25
    λLT = k_c * k_p * λz
        
    λLT0 = 0.4  # Höchstwert (EC3 Seite 70)
    β = 0.75    # Mindestwert
    αLT = KL2α(stab.KL_BDK)
    
    ΦLT = 0.5 * ( 1 + αLT * (λLT - λLT0) + β * λLT**2)
    return min(1 / (ΦLT + sqrt(ΦLT**2 - β * λLT**2)),
               1,
               1 / (λLT**2),
              )

In [13]:
def BDK(um_achse, stab, sg=None):
    last = stab.sg[sg]
    assert(stab.Lcr(um_achse))
    assert(last.ψ is not None)
    χ = χyz(um_achse, stab)
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) ## CHECK past?
    My_Rd =  W_y * (stab.fy / γ_M1) # W_y passt?

    n_y = last.N / (χ * N_Rd)
    k_y = min((1 + (λyz(um_achse, stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    
    C_my = max(0.6 + 0.4 * last.ψ,
               0.4)
    C_MLT = C_my

    n_z = last.N / (χyz('z', stab) * N_Rd)
    λz = λyz('z', stab)
    assert (λz >= 0.4)
    display(f'C_MLT = {C_MLT} λz = {λz}')
    k_LT = 1 - (0.1 * n_z * λz)/(C_MLT - 0.25) ## ??? seite

    interaktionsbeiwert = {'y': k_y * C_my,
                           'z': k_LT,
                          }.get(um_achse)

    res = (last.N/(χ * N_Rd)) + (interaktionsbeiwert * last.My/(χLT(stab) * My_Rd))
    return ausnutzung(N(res, 5), stab)

BDK_y = partial(BDK, 'y')
BDK_z = partial(BDK, 'z')

# Stäbe

In [14]:
staebe = dotdict()

## 1 Pfette

In [15]:
staebe.pfette = Stab(
    S235, HEB240,
    l=6750, Lcry=6750, Lcrz=6750,
    sg = {
        'druck': Schnittgroeszen(
            N  = 118,    # schnee + wind(90)
            My = 73.01,  # schnee + wind(0)
            V = 44.21,  # schnee + wind(0)
            ψ  = 1),
        'zug': Schnittgroeszen(
            N  = 58,     # schnee + wind(90)
            My = 73.01,  # schnee + wind(0)
            V = 44.21,  # schnee + wind(0)
            ψ  = 1),
        },
)

### Biegeknicken (um y)

In [111]:
biege_knicken_y(staebe.pfette, sg='druck')
biege_knicken_z(staebe.pfette, sg='druck')

#### 🔴 245% (HEB 240)

#### 🔴 695% (HEB 240)

### Querschnitt (Druck & Schub)

In [17]:
# TODO

## 2 Binder Obergurt

In [18]:
staebe.obergurt = Stab(
    S235, HEB240,
    l=250.3, Lcry=250.3, Lcrz=250.3,
    sg = {
        'zug': Schnittgroeszen(
            N  = 2158.3, # Schnee leitend + Wind(90)
            My = 78.23,   # Schnee leitend + Wind(0)
            V  = 33.4,   # Schnee leitend + Wind(0)
            ψ  = 8.37/78.23
        ),
        'druck': Schnittgroeszen(
            N  = 279.1,  # Wind(180)
            My = 10.77,   # Wind(180)
            V  = 5.94,    # Wind(180)
            ψ  = 1.40/10.77,
        ),
    })

### Querschnitt (Zug)

In [19]:
kombi_M_N('y', staebe.obergurt, sg='zug')
kombi_M_N('z', staebe.obergurt, sg='zug')

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟢 86% (HEB 240)

Biegung

#### 🟡 0% (HEB 240)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟢 86% (HEB 240)

Biegung

#### 🟡 1% (HEB 240)

### Querschnitt (Druck)

In [20]:
kombi_M_N('y', staebe.obergurt, sg='druck')
kombi_M_N('z', staebe.obergurt, sg='druck')

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 11% (HEB 240)

Biegung

#### 🟡 0% (HEB 240)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 11% (HEB 240)

Biegung

#### 🟡 0% (HEB 240)

### Knicken in Ebene (y)

In [21]:
biege_knicken_y(staebe.obergurt, sg='druck')

#### 🟡 11% (HEB 240)

### Knicken aus der Ebene (z)

In [22]:
biege_knicken_z(staebe.obergurt, sg='druck')

#### 🟡 12% (HEB 240)

## UG Feld

In [23]:
staebe.untergurt_feld = Stab(
    S235, HEB260,
    l=0, Lcry=240, Lcrz=378.1,
    sg = {
        'druck-y': Schnittgroeszen(
            N  = 1707.5,  # Schnee leitend + Wind(90)
            My = 43.1,   # Schnee leitend + Wind(90)
            V  = 31.72,   # Schnee leitend + Wind(90)
            ψ  = 0
        ),
        'druck-z': Schnittgroeszen(
            N  = 2602.14,  # Schnee leitend + Wind(90)
            My = 43.1,   # Schnee leitend + Wind(90)
            V  = 31.72,   # Schnee leitend + Wind(90)
            ψ  = 1,
        ),
        'zug-y': Schnittgroeszen(
            N  = 198,  # Wind(180)
            My = 4.53,   # Wind(180)
            V  = 31.72,    # Wind(180)
            ψ  = 0,
        ),
        'zug-z': Schnittgroeszen(
            N  = 306.9,  # Wind(180)
            My = 4.53,   # Wind(180)
            V  = 31.72,    # Wind(180)
            ψ  = 1,
        ),
    })

### QS-zug

In [24]:
kombi_M_N('y', staebe.untergurt_feld, sg='zug-y')
kombi_M_N('z', staebe.untergurt_feld, sg='zug-z')

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 7% (HEB 260)

Biegung

#### 🟡 0% (HEB 260)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 11% (HEB 260)

Biegung

#### 🟡 0% (HEB 260)

### QS-druck

In [25]:
kombi_M_N('y', staebe.untergurt_feld, sg='druck-y')
kombi_M_N('z', staebe.untergurt_feld, sg='druck-z')

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 61% (HEB 260)

Biegung

#### 🟡 0% (HEB 260)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟢 93% (HEB 260)

Biegung

#### 🟡 0% (HEB 260)

### BDK

In [26]:
BDK_y(staebe.untergurt_feld, sg='druck-y')
BDK_y(staebe.untergurt_feld, sg='druck-z')

'C_MLT = 0.6 λz = 1.92222874551435/pi'

#### 🟡 62% (HEB 260)

'C_MLT = 1.0 λz = 1.92222874551435/pi'

#### 🟢 94% (HEB 260)

## UG Kragarm

In [27]:
staebe.untergurt_krag_li = Stab(
    S235, HEB260,
    l=256, Lcry=256, Lcrz=648.7,
    #l=256, Lcry=256, Lcrz=3.95,
    sg = {
        'druck-y': Schnittgroeszen(
            N  = 1912.6,  # Schnee leitend + Wind(0)
            My = 24.68,   # Schnee leitend + Wind(0)
            V  = 18.62,   # Schnee leitend + Wind(0)
            ψ  = 17.07/24.68,
        ),
        'zug-y': Schnittgroeszen(
            N  = 371.2,  # Wind(180)
            My = 4.77,   # Wind(180)
            V  = 3.45,    # Wind(180)
            ψ  = 3.18/4.77, 
        ),
        'druck-z': Schnittgroeszen(
            N  = 1912.6,  # Schnee leitend + Wind(0)
            My = 24.68,   # Schnee leitend + Wind(0)
            V  = 18.62,   # Schnee leitend + Wind(0)
            ψ  = 0,
        ),
        'zug-z': Schnittgroeszen(
            N  = 371.2,  # Wind(180)
            My = 4.77,   # Wind(180)
            V  = 3.45,    # Wind(180)
            ψ  = 1,
        ),
    })

staebe.untergurt_krag_re = Stab(
    S235, HEB260,
    l=0, Lcry=256, Lcrz=2*485.8,
    #l=0, Lcry=256, Lcrz=2*(485.8 + 255.6),
    sg = {
        'druck-y': Schnittgroeszen(
            N  = 464,  # Schnee leitend + Wind(0)
            My = 9.82,   # Schnee leitend + Wind(0)
            V  = 4.44,   # Schnee leitend + Wind(0)
            ψ  = 2.17/9.82,
        ),
        #'druck-y': Schnittgroeszen(
        #    N  = 891,  # Schnee leitend + Wind(0)
        #    My = 17.1,   # Schnee leitend + Wind(0)
        #    V  = 4.44,   # Schnee leitend + Wind(0)
        #    ψ  = 9.82/17.07,
        #),
        'druck-z': Schnittgroeszen(
            N  = 464,  # Schnee leitend + Wind(0)
            My = 9.82,   # Schnee leitend + Wind(0)
            V  = 4.44,   # Schnee leitend + Wind(0)
            ψ  = 0
        ),
        #'druck-z': Schnittgroeszen(
        #    N  = 891,  # Schnee leitend + Wind(0)
        #    My = 17.1,   # Schnee leitend + Wind(0)
        #    V  = 4.44,   # Schnee leitend + Wind(0)
        #    ψ  = 0,
        #),
    })
note('ψ werte fuer -z stab doublechecken!!')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">ψ werte fuer -z stab doublechecken!!</span></div>

### QS-zug

In [28]:
md('#### Kargarm (zug) in Ebene')
kombi_M_N('y', staebe.untergurt_krag_li, sg='zug-y')

md('#### Kargarm (zug) aus Ebene')
kombi_M_N('z', staebe.untergurt_krag_li, sg='zug-y')

#### Kargarm (zug) in Ebene

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 13% (HEB 260)

Biegung

...

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 13% (HEB 260)

Biegung

#### 🟡 0% (HEB 260)

### QS-druck

In [29]:
md('#### Kragarm (druck) in Ebene')
kombi_M_N('y', staebe.untergurt_krag_li, sg='druck-y')
kombi_M_N('y', staebe.untergurt_krag_re, sg='druck-y')

md('#### Kragarm (druck) aus Ebene')
kombi_M_N('z', staebe.untergurt_krag_li, sg='druck-z')
kombi_M_N('z', staebe.untergurt_krag_re, sg='druck-z')

#### Kragarm (druck) in Ebene

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 68% (HEB 260)

Biegung

...

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 16% (HEB 260)

Biegung

#### 🟡 0% (HEB 260)

### BDK-y

In [30]:
BDK_y(staebe.untergurt_krag_li, sg='druck-y')
BDK_y(staebe.untergurt_krag_re, sg='druck-y')


'C_MLT = 0.8766612641815235 λz = 3.29793649091552/pi'

#### 🟡 69% (HEB 260)

'C_MLT = 0.6883910386965376 λz = 4.93953305776711/pi'

#### 🟡 17% (HEB 260)

### BDK-z

In [31]:
BDK_z(staebe.untergurt_krag_li, sg='druck-z')
BDK_z(staebe.untergurt_krag_re, sg='druck-z')


'C_MLT = 0.6 λz = 3.29793649091552/pi'

#### 🔴 134% (HEB 260)

'C_MLT = 0.6 λz = 4.93953305776711/pi'

#### 🟡 57% (HEB 260)

## Streben FW

In [35]:
staebe.strebe = Stab(
    S235, HEA140,
    l=192, Lcry=192, Lcrz=192,
    sg = {
        'druck': Schnittgroeszen(
            N  = 666, # Schnee leitend + Wind(0)
            My = 0,
            V  = 0,
        ),
        'zug': Schnittgroeszen(
            N  = 794,  # Wind(180)
            My = 0,
            V  = 0,
        ),
    }
)


### Querschnitt (Zug)

In [36]:
zug(staebe.strebe, sg='zug')

#### 🔴 107% (HEA 140)

### Knicken in Ebene

In [42]:
## Druck: Knicken aus Ebene
# Maßgeblicher Stab:
# LF: Schnee leitend + Wind (0)
biege_knicken_y(staebe.strebe, sg='druck')

#### 🟢 95% (HEA 140)

### Knicken aus der Ebene

In [43]:
## Druck: Knicken aus Ebene
# Maßgeblicher Stab:
# LF: Schnee leitend + Wind (0)
biege_knicken_z(staebe.strebe, sg='druck')

#### 🔴 113% (HEA 140)

## Streben UG

In [44]:
staebe.strebe_UG = Stab(
    S235, HEB240, l=149, Lcry=149, Lcrz=149,
    sg = {
        'druck': Schnittgroeszen(
            N  = 1260,  # Schnee leitend + Wind(0)
            My = 0,
            V  = 0,
        ),
        'zug': Schnittgroeszen(
            N  = 150,  # Wind(180)
            My = 0,
            V  = 0,
        ),
    }
)

### Querschnitt (Zug)

In [52]:
zug(staebe.strebe_UG, sg='zug')

#### 🟡 6% (HEB 240)

### Knicken in Ebene

In [53]:
## Druck: Knicken aus Ebene
# Maßgeblicher Stab:
# LF: Schnee leitend + Wind (0)
biege_knicken_y(staebe.strebe_UG, sg='druck')

#### 🟡 50% (HEB 240)

### Knicken aus der Ebene

In [54]:
## Druck: Knicken aus Ebene
# Maßgeblicher Stab:
# LF: Schnee leitend + Wind (0)
biege_knicken_z(staebe.strebe_UG, sg='druck')

#### 🟡 52% (HEB 240)

## Dachverband

In [67]:
staebe.dachverband = Stab(
    S235, RD30, l=0, Lcry=0, Lcrz=0,
    sg = Schnittgroeszen(
        N  = 155,  # schnee + wind(90)
        My = 0,
        V  = 0,
    ),
)

### Querschnitt (Zug)

In [66]:
zug(staebe.dachverband)

#### 🟢 93% (RD 30)

## 6 Wandverband

In [77]:
staebe.wandverband = Stab(
    S235, RD45, l=0, Lcry=0, Lcrz=0,
    sg = Schnittgroeszen(
        N  = 341,  # Wind(180)
        My = 0,
        V  = 0,
    ),
)

### Querschnitt (Zug)

In [78]:
# TODO
zug(staebe.wandverband)

#### 🟢 91% (RD 45)

## 7 Stütze A

In [79]:
staebe.stuetzeA = Stab(
    S235, HEA140, l=650, Lcry=650, Lcrz=650,
    sg = {
        'zug': Schnittgroeszen(
            N  = 705,    # Schnee + wind(90)
            My = 0,
            V  = 0,
        ),
        'druck': Schnittgroeszen(
            N  = 84,  # Wind (180)
            My = 0,
            V  = 0,
        ),
    })

### Querschnitt (Zug)

In [80]:
zug(staebe.stuetzeA, sg='zug')

#### 🟢 95% (HEA 140)

### Querschnitt (Druck)

In [81]:
druck(staebe.stuetzeA, sg='druck')

#### 🟡 11% (HEA 140)

### Knicken in Ebene

In [82]:
biege_knicken_y(staebe.stuetzeA, sg='druck')

#### 🟡 24% (HEA 140)

### Knicken aus der Ebene

In [83]:
biege_knicken_z(staebe.stuetzeA, sg='druck')

#### 🟡 56% (HEA 140)

## 8 Stütze B

In [105]:
staebe.stuetzeB = Stab(
    S235, HEB340, l=775, Lcry=0, Lcrz=625,
    sg = {
        'druck-y': Schnittgroeszen(
            N  = 1366,  # schnee + wind(0)
            My = 1073,
            V  = 716,
            ψ  = 1,
        ),
        'druck-z': Schnittgroeszen(
            N  = 1366,  # schnee + wind(0)
            My = 1073,
            V  = 176,
            ψ  = 0,
        ),
        'zug': Schnittgroeszen(
            N  = 705,    # Schnee + wind(90)
            My = 0,
            V  = 0,
        ),
    })

### Knicklänge (um y)

In [106]:
h_Fachwerk = 108                 # Achsabstand der Gurte in Mitte des Feldbereichs vom Fachwerk
z = h_Fachwerk * 0.5
l_R = 500
l = 775
l1 = staebe.stuetzeA.l

N_A = abs(staebe.stuetzeA.sg['zug'].N)
N_B = abs(staebe.stuetzeB.sg['druck-y'].N)
F = N_A * (l / l1)   # <--- Einsparungspotential!  ggf hoehe bis traegermitte waehlbar


EI = staebe.stuetzeB.E * staebe.stuetzeB.Iy
EI_R = staebe.obergurt.E * ((staebe.obergurt.Iy + staebe.obergurt.A * z**2) + (staebe.untergurt_feld.Iy + staebe.untergurt_feld.A * z**2))

display(staebe.stuetzeA.l)
display(EI_R)

ν = 1 / ( (l_R * EI)/(l * EI_R) + 1)
η = ( (3 * ν) / ( (0.216 * ν**2 + 1) * N_B + F)) * (EI/l**2)
staebe.stuetzeB.Lcry = N(pi * sqrt(EI/(η*N_B)))
staebe.stuetzeB

Stab(guete='S235', E=21000, fy=23.5, fu=36, querschnitt='HEB 340', A=170.9, Avy=133.68, Avz=56.09, b=30, h=34, tf=2.15, Iy=36660, Iz=9690, iy=14.65, iz=7.53, KLy='b', KLz='c', l=775, sg={'druck-y': Schnittgroeszen(N=1366, V=716, My=1073, Mz=None, ψ=1), 'druck-z': Schnittgroeszen(N=1366, V=176, My=1073, Mz=None, ψ=0), 'zug': Schnittgroeszen(N=705, V=0, My=0, Mz=None, ψ=None)}, Lcry=1927.51564843306, Lcrz=625)

### Querschnitt (Zug)

In [107]:
kombi_M_N('y', staebe.stuetzeB, sg='zug')
kombi_M_N('z', staebe.stuetzeB, sg='zug')

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 17% (HEB 340)

Biegung

#### 🟡 0% (HEB 340)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 17% (HEB 340)

Biegung

#### 🟡 0% (HEB 340)

### Querschnitt (Druck)

In [108]:
kombi_M_N('y', staebe.stuetzeB, sg='druck-y')
kombi_M_N('z', staebe.stuetzeB, sg='druck-z')

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 34% (HEB 340)

Biegung

#### 🟡 2% (HEB 340)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 34% (HEB 340)

Biegung

#### 🟡 7% (HEB 340)

### BDK in Ebene

In [109]:
BDK_y(staebe.stuetzeB, sg='druck-y')

'C_MLT = 1.0 λz = 2.77657446241755/pi'

#### 🟢 93% (HEB 340)

### Knicken aus der Ebene

In [110]:
biege_knicken_z(staebe.stuetzeB, sg='druck-z')

#### 🟡 62% (HEB 340)